# Hmall 협력사 문서 RAG 챗봇 — 미니 프로젝트 1

현대 Hmall 협력사 대상 안내 문서 7종(총 138p)을 근거로 답변하는 RAG 시스템.

| # | 문서 | 페이지 |
|---|---|---:|
| 1 | 협력사 운영 통합 안내서 | 23 |
| 2 | Hmall 소개서 | 30 |
| 3 | 쇼라 소개서 | 22 |
| 4 | 광고 상품 소개서 | 24 |
| 5 | 협력사시스템 사용법 | 10 |
| 6 | 신규 협력사 입점 절차 안내 | 21 |
| 7 | 데이터 영역 광고 제안서 | 8 |

### 파이프라인

| 단계 | 내용 | 핵심 선택 |
|---|---|---|
| 1 | 데이터 수집·Ingestion | 하이브리드 VLM 파싱 (페이지 이미지 + 텍스트 레이어) |
| 2 | 골든 데이터셋 생성·큐레이션 | 20문항 / 2단계 자동 검수 / 무답변 문항 자동 검증 |
| 3 | RAG 검색·생성 워크플로우 | Chroma + LCEL, 표 행 조건 준수 프롬프트 |
| 4 | 성능 평가 | Ragas 표준 4지표 + 커스텀 2지표 |
| 5 | 챗봇 | Streamlit (`app.py`) |

### 이 문서 집합의 특징

7개 모두 "텍스트 문서"가 아니라 **프레젠테이션·브로슈어를 PDF로 내보낸 자료**다.
표가 격자로 묶여 있지 않고, 일부 문서는 텍스트 레이어 자체가 없다.
이 점이 1단계 파서 선택을 좌우했고, 그대로 2단계 문항 구성과 4단계 평가지표로 이어진다.


In [ ]:
# =====================================================================
# 0. 공통 설정
# =====================================================================
# 이후 모든 단계가 이 셀의 모델명·임베딩 설정을 공유한다.
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.callbacks import UsageMetadataCallbackHandler

load_dotenv(find_dotenv(".env", usecwd=True), override=True)

MODEL     = os.environ["OPENAI_DEFAULT_MODEL"]    # 파싱·생성용 상위 모델
LOWER     = os.environ["OPENAI_LOWER_MODEL"]      # 단순 페이지 파싱용 경량 모델
JUDGE     = os.environ["OPENAI_JUDGE_MODEL"]      # 검수·평가 심사위원
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]

CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "langchain"
GOLDEN_PATH     = "golden_dataset.json"

usage = UsageMetadataCallbackHandler()   # 토큰 사용량 집계용
embeddings = OpenAIEmbeddings(model=EMBEDDING)

print(f"MODEL={MODEL} / LOWER={LOWER} / JUDGE={JUDGE} / EMBEDDING={EMBEDDING}")

---
# 1단계: 데이터 수집 및 Ingestion

## PDF 로더 선정 — 측정 근거와 최종 결론

로더를 고르기 전에 7개 PDF의 실제 상태를 먼저 측정했다.

| 파일 | 페이지 | 텍스트 레이어 | 이미지 | `find_tables()` |
|---|---:|---:|---:|---:|
| 1. 협력사 운영 통합 안내서 | 23 | 8,224자 | 150 | 21 |
| 2. Hmall 소개서 | 30 | 6,206자 | 79 | 11 |
| **3. 쇼라 소개서** | 22 | **0자** | 22 | 0 |
| 4. 광고 상품 소개서 | 24 | 8,625자 | 123 | 24 |
| **5. 협력사시스템 사용법** | 10 | **6자** | 66 | 0 |
| 6. 신규 협력사 입점 절차 안내 | 21 | 6,473자 | 139 | 2 |
| 7. 데이터 영역 광고 제안서 | 8 | 994자 | 17 | 4 |

### 진단

**① 쇼라 소개서는 로더를 바꿔도 해결되지 않는다.**
전 페이지가 1920×1080 이미지 1장으로만 구성된 스캔형 PDF다(텍스트 레이어 0자).
PyPDF·PyMuPDF·pdfplumber는 전부 텍스트 레이어를 읽는 방식이라 어느 것을 써도 결과는 0자다.
→ **로더 교체 문제가 아니라 OCR/비전 문제**. 협력사시스템 사용법(6자)도 같은 경우.

**② 광고 상품 소개서의 표는 진짜 표가 아니다.**
셀이 격자로 묶여 있지 않고 개별 텍스트박스 + 벡터 도형으로 그려져 있다.
p.14 단가표에 세 로더를 적용한 결과가 모두 실패했다.

- `get_text("text")` → 셀이 읽기 순서대로 쏟아져 행·열 대응 완전 소실
- `pdfplumber.extract_tables()` → `['구', '분 상품명', None, ...]` 로 셀 병합이 어긋남. 복잡한 셀 병합이 들어간 표는 텍스트를 추출했을때 의미를 복원 할 수 없음
- `pymupdf4llm` → 표를 그림으로 인식해 picture text로 밀어냄
- `PyMuPDF.find_tables(strategy="lines")` → 검출된 표 **0개**

**③ RapidOCR 기본 모델은 한글을 지원하지 않는다.**
쇼라 p.6 OCR 결과가 `03. 50 202222023314 ... 喜2号PD（17）` 로 한글이 한자·숫자로 깨졌다.

### 최종 선택: 하이브리드 VLM 파싱 (페이지 이미지(A) + 텍스트 레이어(B))

페이지 이미지(A): 표/차트/배치 구조 파악. 페이지 이미지를 보고 시각적으로 인식

텍스트 레이어(B): 정확한 텍스트/철자. 위치 정보/순서 없음

구조는 A, 글자는 B
정확도가 보장된 마크다운 + 표 행 풀어쓰기 문장

페이지를 200 DPI 이미지로 렌더링해 비전 모델에 넘기되, **PDF 텍스트 레이어를 함께 첨부**한다.

- **이미지** → 표의 행·열 구조와 배치를 읽는 용도
- **텍스트 레이어** → 철자의 정답지

두 번째가 핵심이다. 이미지만 줬을 때는 작은 한글을 `현대백화점탭 → 백화점법`,
`투뎁스 → 투탑스`, `구좌수(日) → 구좌수(타)` 로 오독했고 DPI를 300으로 올려도 고쳐지지 않았다.
그런데 **그 단어들은 텍스트 레이어에 정확히 들어 있었다.** 구조는 이미지에서, 철자는 텍스트
레이어에서 가져오게 하자 오독이 사라졌다. 두 소스의 강점이 정확히 상보적이다.

추가로 표를 마크다운 테이블과 **`#### 표 행 풀어쓰기`(행 단위 문장)** 두 형태로 받는다.
청킹 과정에서 표가 중간에 잘려도 행 단위 문장은 행·열 대응이 그대로 살아남아,
4단계 평가지표인 **표 행/열 매핑 정합성**에 직접 기여한다.

### 모델 라우팅

같은 프롬프트로 p.14 단가표를 두 모델에 돌려 비교했다.

| | 열 구조 | 철자 | 병합 셀 그룹 경계 |
|---|---|---|---|
| gpt-5.4-mini | ✗ 헤더 5열 / 데이터 6열 불일치 | ○ | ✗ D·E를 카테고리로 잘못 묶음 |
| **gpt-5.4** | **○ 6열 일치** | **○** | **○ 정확** |

mini는 열 수를 맞추려고 데이터 값을 지워버리는 실패를 반복했다. 따라서
**표가 있거나 스캔형(텍스트 레이어 100자 미만)인 페이지는 gpt-5.4, 나머지는 gpt-5.4-mini**로
라우팅해 정확도와 비용을 함께 잡았다.

### 후보 비교 요약

| 후보 | 쇼라(스캔) | 표 행/열 | 이미지 내 글자 | 채택 |
|---|---|---|---|---|
| PyPDF | ✗ 0자 | ✗ | ✗ | |
| PyMuPDF `get_text` | ✗ 0자 | ✗ | ✗ | |
| pdfplumber | ✗ 0자 | △ 병합 깨짐 | ✗ | |
| pymupdf4llm + RapidOCR | △ 한글 깨짐 | ✗ | △ 깨짐 | |
| VLM 단독 (이미지만) | ○ | △ 철자 오독 | ○ | |
| **VLM + 텍스트 레이어** | **○** | **○** | **○** | **✔** |

트레이드오프는 비용과 처리 시간(138페이지 1회)이다.
**페이지 단위 디스크 캐시(`md_cache/`)** 로 재실행 비용을 0으로 만들어 이를 상쇄했다.

### 청킹 전략

`MarkdownHeaderTextSplitter`(`#`, `##`) 로 1차 분할해 문서의 계층 구조를 메타데이터로 보존하고,
`RecursiveCharacterTextSplitter`(900자 / overlap 150)로 2차 분할한다.
`####` 는 분할 기준에서 뺐다 — `#### 표 행 풀어쓰기`가 표 본문과 떨어지면 안 되기 때문이다.

### 결과

| 파일 | 기존 `get_text` | 하이브리드 VLM |
|---|---:|---:|
| 3. 쇼라 소개서 | **0자** | **15,110자** |
| 5. 협력사시스템 사용법 | **6자** | **13,090자** |
| 4. 광고 상품 소개서 | 8,625자 | 26,700자 |
| **전체** | **30,528자** | **115,669자** |


In [ ]:
# =====================================================================
# 1단계(개정): 하이브리드 VLM 기반 PDF 파싱 파이프라인
# ---------------------------------------------------------------------
# [기존 방식의 문제]
# 대상 7개 PDF는 "텍스트 문서"가 아니라 프레젠테이션/브로슈어를 내보낸 자료다.
#   - '3. 쇼라 소개서'(22p)는 전 페이지가 1920x1080 이미지 1장 = 텍스트 레이어 0자
#   - '5. 협력사시스템 사용법'(10p)도 텍스트 레이어 6자
#     => PyPDF/PyMuPDF/pdfplumber는 모두 텍스트 레이어를 읽으므로 결과가 반드시 0자
#   - '4. 광고 상품 소개서'의 표는 격자로 묶인 표가 아니라 개별 텍스트박스 + 벡터 도형
#     => get_text는 셀이 읽기 순서대로 쏟아져 행/열 대응 소실,
#        pdfplumber는 셀 병합이 어긋나 깨짐, pymupdf4llm은 표를 그림으로 처리
#   - RapidOCR 기본 모델은 중/영문용이라 한글이 한자로 깨짐
#
# [채택 방식] 페이지 이미지 + PDF 텍스트 레이어를 함께 VLM에 넘긴다.
#   - 이미지 : 표의 행/열 구조와 배치를 읽는 용도
#   - 텍스트 레이어 : 철자의 정답지 (VLM 단독 판독 시 '현대백화점탭'->'백화점법',
#                     '투뎁스'->'투탑스' 로 오독되는 것을 텍스트 레이어가 교정)
#   - 표는 마크다운 테이블 + "행 풀어쓰기" 두 형태로 받는다.
#     청킹 과정에서 표가 잘려도 행 단위 문장은 행/열 대응이 살아남아
#     본 프로젝트 평가지표인 '표 행/열 매핑 정합성'에 직접 기여한다.
# =====================================================================
import os, glob, base64, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pymupdf
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
client = OpenAI()

STRONG_MODEL = os.environ["OPENAI_DEFAULT_MODEL"]   # 표/스캔 페이지용
LIGHT_MODEL  = os.environ["OPENAI_LOWER_MODEL"]     # 단순 텍스트 페이지용
EMBEDDING    = os.environ["OPENAI_EMBEDDING_MODEL"]

FILES_DIR = Path.cwd() / "files"
CACHE_DIR = Path.cwd() / "md_cache"    # 페이지별 캐시 -> 재실행 시 API 비용 0
CACHE_DIR.mkdir(exist_ok=True)
DPI = 200

PROMPT = """아래는 같은 PDF 페이지의 두 가지 정보다.
(A) 페이지 이미지 - 표의 행/열 구조와 화면 배치를 여기서 읽는다.
(B) PDF 내부 텍스트 레이어 - 글자 표기(철자)가 정확하다. 단 순서가 뒤섞여 구조 정보는 없다.

규칙:
1. 구조는 (A)에서, 글자 표기는 (B)에서 가져온다. (B)에 있는 단어는 (B)의 철자를 그대로 쓰고
   이미지로 읽은 추측 철자로 덮어쓰지 않는다. ('탭/법', '뎁스/탑스', '日/타' 등 혼동 금지)
2. 표가 있으면 마크다운 파이프 테이블로 출력한다.
   - 열 개수는 데이터 행을 기준으로 정한다. 머리글이 없는 열(하위 구분 등)이 있으면
     헤더에 이름을 새로 지어 붙여서 열을 늘린다. 값을 지워 열 수를 맞추는 것은 절대 금지.
   - 세로로 병합된 셀은 이미지에서 그 셀이 세로로 덮고 있는 행 범위를 정확히 확인해서
     해당하는 모든 행에 값을 반복해 채운다. 그룹 경계를 틀리지 마라.
3. 표 아래에 `#### 표 행 풀어쓰기` 를 붙이고, 각 데이터 행을 한 문장씩 쓴다.
   모든 열의 값이 문장 안에 빠짐없이 들어가야 하고, 데이터 행 수와 문장 수가 같아야 한다.
4. 제목은 #, 소제목은 ##. 화면 캡처/차트/흐름도 안의 글자도 모두 옮기고,
   절차 흐름도는 "1단계 -> 2단계" 형태로 순서를 풀어 쓴다.
5. 금액·기간·수수료율·URL·날짜는 원문 표기 그대로 옮긴다.
6. 해설/요약/추측 금지. 페이지에 실제로 적힌 내용만 출력한다.

=== (B) 텍스트 레이어 ===
{layer}
=== 끝 ==="""


def pick_model(page, layer: str) -> str:
    """표가 있거나 스캔형(텍스트 레이어 없음) 페이지는 구조 복원이 어려우므로 상위 모델."""
    if len(layer.strip()) < 100:              # 스캔 이미지 페이지
        return STRONG_MODEL
    try:
        if len(page.find_tables().tables) > 0:  # 표 포함 페이지
            return STRONG_MODEL
    except Exception:
        pass
    return LIGHT_MODEL


def parse_page(args):
    pdf_path, page_no = args
    with pymupdf.open(pdf_path) as doc:
        page = doc[page_no]
        layer = page.get_text("text")
        model = pick_model(page, layer)
        png = page.get_pixmap(dpi=DPI).tobytes("png")

    key = hashlib.md5(f"{os.path.basename(pdf_path)}|{page_no}|{model}|{DPI}|v2".encode()).hexdigest()[:16]
    cache_file = CACHE_DIR / f"{key}.md"
    if cache_file.exists():
        return pdf_path, page_no, model, cache_file.read_text(encoding="utf-8")

    b64 = base64.b64encode(png).decode()
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": PROMPT.format(
                layer=layer if layer.strip() else "(텍스트 레이어 없음 - 이미지만으로 판독)")},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"}},
        ]}],
    )
    md = (resp.choices[0].message.content or "").strip()
    cache_file.write_text(md, encoding="utf-8")
    return pdf_path, page_no, model, md


# ---------- 1. 전 페이지 파싱 ----------
pdf_files = sorted(glob.glob(str(FILES_DIR / "*.pdf")))
jobs = []
for p in pdf_files:
    with pymupdf.open(p) as d:
        jobs += [(p, i) for i in range(d.page_count)]

print(f"📄 대상 PDF {len(pdf_files)}개 / 총 {len(jobs)}페이지 파싱 시작")
with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(parse_page, jobs))

# ---------- 2. 페이지 단위 Document ----------
raw_docs, per_file, model_use = [], {}, {}
for pdf_path, page_no, model, md in sorted(results, key=lambda r: (r[0], r[1])):
    name = os.path.basename(pdf_path)
    per_file[name] = per_file.get(name, 0) + len(md)
    model_use[model] = model_use.get(model, 0) + 1
    if md.strip():
        raw_docs.append(Document(page_content=md,
                                 metadata={"source": name, "page": page_no + 1}))

print(f"\n✅ 파싱 완료: {len(raw_docs)}페이지  (모델 사용: {model_use})")
for name, n in sorted(per_file.items()):
    print(f"   {name[:34]:36s} {n:7,d}자")

# ---------- 3. 청킹 ----------
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2")],   # #### 표 행 풀어쓰기는 표와 붙여 둔다
    strip_headers=False,
)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""],
)

header_docs = []
for d in raw_docs:
    for s in header_splitter.split_text(d.page_content):
        s.metadata = {**d.metadata, **s.metadata}
        header_docs.append(s)

final_chunks = [c for c in text_splitter.split_documents(header_docs)
                if len(c.page_content.strip()) >= 30]   # 표 잔여 조각 등 노이즈 제거
print(f"\n✨ 최종 청크: {len(final_chunks)}개 "
      f"(평균 {sum(len(c.page_content) for c in final_chunks)//max(len(final_chunks),1)}자)")

# ---------- 4. Chroma 재구축 ----------
persist_directory = "./chroma_db"
embeddings = OpenAIEmbeddings(model=EMBEDDING)

# 구버전 파싱 결과가 섞이지 않도록 컬렉션을 비우고 다시 적재
Chroma(persist_directory=persist_directory,
       embedding_function=embeddings).reset_collection()

vectorstore = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
)
print(f"🚀 Chroma 재구축 완료 → {persist_directory} ({vectorstore._collection.count()}개 벡터)")

---
# 2단계: 골든 데이터셋 생성 및 큐레이션

총 20문항. **1단계에서 표 파싱이 핵심 개선점이었으므로 `표/절차`를 3 → 6문항으로 확대**했다.

| 유형 | 문항 | 의도 |
|---|---:|---|
| 단일홉 | 4 | 근거 한 곳으로 답하는 기본 검색 |
| 멀티홉 | 3 | 서로 다른 페이지 두 곳을 결합해야 답이 나옴 |
| **표/절차** | **6** | **표에서 행 조건에 맞는 값을 골라내는가 (1단계 개선의 직접 검증)** |
| 부정/예외 | 2 | 제외·불가 조건 |
| 시점/기한 | 2 | 기한·처리기간·D-day |
| 무답변 | 3 | 문서에 없는 질문에 지어내지 않는가 |

### 설계 포인트

**① 표 문항의 근거를 키워드가 아니라 표 구조로 지목한다.**
1단계 파싱이 표를 마크다운 테이블 + 행 풀어쓰기로 남기므로,
`"|---" in chunk` 로 표 청크를 정확히 골라낼 수 있다(표 포함 141개 / 일반 82개).
문서별 라운드로빈으로 뽑아 한 문서에 문항이 쏠리지 않게 했다.

**② 표 문항은 "행 조건 2개 이상"을 강제한다.**
`구분`과 `하위구분`을 모두 지정해야 답이 하나로 특정되는 질문만 만들게 했다.
표 전체를 나열시키는 질문은 행/열 매핑 능력을 측정하지 못한다.

**③ 목차·표지 청크를 근거에서 제외한다.**
목차는 '항목명 → 페이지번호' 나열일 뿐이라
"몇 페이지를 보면 되는가" 같은 무의미한 문항이 만들어진다.

**④ 2단계 자동 검수**
- 1차 **인용 정합성**: 초안의 인용구가 근거에 **한 글자도 다르지 않게** 존재하는지 검사.
  멀티홉은 근거가 2개이므로 인용구도 근거별로 따로 받아 각각 대조한다
  (한 문자열로 합치게 하면 모델이 `/`로 이어붙여 어느 근거와도 일치하지 않는다).
- 2차 **자기완결성**: 질문만 단독으로 읽어도 의도가 특정되는지 LLM이 판정.

**⑤ 무답변 문항을 하드코딩하지 않고 검증한다.**
1단계 개선으로 회수 텍스트가 3.8배 늘어, 예전에 "문서에 없던" 질문이 이제는 답이 나올 수 있다.
후보 질문마다 검색을 돌려 상위 청크로 답이 가능한지 심사하고, **불가 판정된 것만** 채택한다.


In [ ]:
# =====================================================================
# 2단계: 골든 데이터셋 생성 및 큐레이션
# ---------------------------------------------------------------------
# 목표 20문항. 이번 개정에서 표 파싱이 핵심 개선점이므로 '표/절차'를 3 -> 6으로 확대했다.
# 1단계 파싱이 표를 마크다운 테이블 + '표 행 풀어쓰기' 형태로 남기므로,
# 표 문항의 근거 청크를 키워드 추측이 아니라 실제 표 구조로 정확히 지목할 수 있다.
# =====================================================================
import os, re, json, random
from typing import Literal, List
from pydantic import BaseModel, Field

from dotenv import load_dotenv, find_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL     = os.environ["OPENAI_DEFAULT_MODEL"]
JUDGE     = os.environ["OPENAI_JUDGE_MODEL"]
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]

CHROMA_PATH, COLLECTION_NAME = "./chroma_db", "langchain"
random.seed(42)

# 표/절차 확대 배분 (총 20)
TARGET = {"단일홉": 4, "멀티홉": 3, "표/절차": 6, "부정/예외": 2, "시점/기한": 2, "무답변": 3}
OVERSAMPLE = 3
print(f"🎯 목표 {sum(TARGET.values())}문항 / 유형별 {TARGET} (오버샘플 {OVERSAMPLE}배)")

# ---------- 1. 청크 로드 ----------
print("\n[1/5] Chroma DB 로드")
embeddings = OpenAIEmbeddings(model=EMBEDDING)
vectorstore = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings,
                     collection_name=COLLECTION_NAME)
data = vectorstore.get()
TOC_PAT = re.compile(r"^#+\s*(목차|차례|INDEX|CONTENTS|Ⅰ\.\s*회사개요)", re.I | re.M)

def is_toc(text: str) -> bool:
    """목차·표지 청크 제외.
    목차는 '항목명 -> 페이지번호' 나열일 뿐 사실 정보가 없어서
    질문을 만들면 '몇 페이지를 보면 되는가' 같은 무의미한 문항이 나온다."""
    if TOC_PAT.search(text):
        return True
    # '... 페이지는 6P이다' 류가 3줄 이상이면 목차 표로 간주
    return len(re.findall(r"페이지는\s*\d+\s*P", text)) >= 3

body, skipped_toc = [], 0
for t, m in zip(data["documents"], data["metadatas"]):
    if not t or len(t.strip()) <= 150:
        continue
    if is_toc(t):
        skipped_toc += 1
        continue
    body.append(Document(page_content=t, metadata=m or {}))
print(f"✅ 유효 본문 청크 {len(body)}개 (목차/표지 {skipped_toc}개 제외)")

# ---------- 2. 유형별 근거 샘플링 ----------
print("\n[2/5] 유형별 근거 청크 샘플링")

def has_table(c):
    """1단계 파싱이 만든 마크다운 표 또는 행 풀어쓰기를 가진 청크."""
    return "|---" in c.page_content or "표 행 풀어쓰기" in c.page_content

def having_any(keywords, n, pool=None):
    matched = [c for c in (pool or body) if any(k in c.page_content for k in keywords)]
    random.shuffle(matched)
    return matched[:n]

table_pool = [c for c in body if has_table(c)]
plain_pool = [c for c in body if not has_table(c)]
print(f"   표 포함 청크 {len(table_pool)}개 / 일반 청크 {len(plain_pool)}개")

# 단일홉: 일반 청크에서 균등 간격
step = max(1, len(plain_pool) // (TARGET["단일홉"] * OVERSAMPLE))
single_chunks = plain_pool[::step][:TARGET["단일홉"] * OVERSAMPLE]

# 멀티홉: 유사도 검색으로 연관 청크 짝짓기
pairs = []
for i in range(0, len(body), max(1, len(body) // (TARGET["멀티홉"] * OVERSAMPLE))):
    base = body[i]
    # 같은 페이지 조각끼리 묶으면 '멀티홉'이 되지 않으므로 다른 페이지에서 짝을 찾는다
    partner = next((d for d in vectorstore.similarity_search(base.page_content, k=8)
                    if d.page_content != base.page_content
                    and d.metadata.get("page") != base.metadata.get("page")), None)
    if partner:
        pairs.append((base, partner))
pairs = pairs[:TARGET["멀티홉"] * OVERSAMPLE]

# 표/절차: 표 청크를 직접 사용 (여러 문서에 고루 퍼지도록 source 기준 라운드로빈)
by_source = {}
for c in table_pool:
    by_source.setdefault(c.metadata.get("source", "?"), []).append(c)
for v in by_source.values():
    random.shuffle(v)
table_chunks, idx = [], 0
while len(table_chunks) < TARGET["표/절차"] * OVERSAMPLE:
    added = False
    for v in by_source.values():
        if idx < len(v):
            table_chunks.append(v[idx]); added = True
            if len(table_chunks) >= TARGET["표/절차"] * OVERSAMPLE:
                break
    if not added:
        break
    idx += 1

negation_chunks = having_any(["제외", "불가", "않습니다", "없습니다", "단,", "유의"],
                             TARGET["부정/예외"] * OVERSAMPLE, plain_pool)
temporal_chunks = having_any(["기한", "까지", "이내", "D-", "영업일", "익일", "마감"],
                             TARGET["시점/기한"] * OVERSAMPLE)

jobs  = [("단일홉", [c]) for c in single_chunks]
jobs += [("멀티홉", list(p)) for p in pairs]
jobs += [("표/절차", [c]) for c in table_chunks]
jobs += [("부정/예외", [c]) for c in negation_chunks]
jobs += [("시점/기한", [c]) for c in temporal_chunks]
print(f"   생성할 초안 작업 {len(jobs)}개")

# ---------- 3. 초안 생성 ----------
print("\n[3/5] 초안 생성")

class Draft(BaseModel):
    question: str = Field(description="질문만 읽어도 무엇을 묻는지 특정되어야 함. '이 규정', '해당 문서' 같은 지시대명사 금지.")
    answer: str   = Field(description="근거에 기반한 정확한 정답")
    quotes: List[str] = Field(
        description="근거에서 **한 글자도 바꾸지 않고 그대로 복사한** 문장들. "
                    "근거가 2개면 각 근거에서 1개씩 총 2개. 여러 문장을 '/'나 따옴표로 "
                    "이어 붙이지 말고 리스트의 별도 항목으로 담아라. 요약·재작성 금지.")
    difficulty: Literal["easy", "medium", "hard"]

GUIDES = {
    "단일홉": "근거 한 곳만 보면 답할 수 있는 명확한 질문을 만든다.",
    "멀티홉": "두 근거를 모두 결합해야만 답할 수 있는 질문을 만든다. 한쪽만 봐서는 답이 나올 수 없어야 한다.",
    "표/절차": ("표에서 **특정 행과 특정 열이 교차하는 값**을 묻는 질문을 만든다. "
              "예: '어떤 구분의 어떤 상품의 단가는 얼마인가'처럼 행을 특정하는 조건을 2개 이상 넣어라. "
              "표 전체를 나열시키는 질문이나 열 이름만 묻는 질문은 만들지 마라. "
              "절차 표라면 특정 단계에서 무엇을 하는지 묻는다. "
              "값이 비어 있는 셀은 절대 묻지 마라 — 정답이 '빈칸'이 되는 질문은 금지."),
    "부정/예외": "무엇을 하면 안 되는지, 어떤 예외·제외 조건이 있는지를 묻는 질문을 만든다.",
    "시점/기한": "신청 기한, 처리 기간, D-day, 영업일 등 시간 조건을 묻는 질문을 만든다.",
}

writer_chain = ChatPromptTemplate.from_messages([
    ("system", "제공된 규정/매뉴얼 문서를 근거로 평가용 질문과 답을 작성합니다. "
               "근거에 없는 사실은 절대 작성하지 마세요.\n{guide}"),
    ("human", "근거 문서:\n{context}"),
]) | ChatOpenAI(model=MODEL, temperature=0.2).with_structured_output(Draft)

drafts = []
for kind, cs in jobs:
    try:
        d = writer_chain.invoke({"guide": GUIDES[kind],
                                 "context": "\n---\n".join(c.page_content for c in cs)})
        drafts.append((kind, cs, d))
    except Exception:
        continue
print(f"   초안 {len(drafts)}개 생성")

# ---------- 4. 2단계 검수 ----------
print("\n[4/5] 검수")
squash = lambda t: re.sub(r"\s+", "", t)

stage1, dropped = [], []
for kind, cs, d in drafts:
    source = squash(" ".join(c.page_content for c in cs))
    qs = [squash(q) for q in d.quotes if len(squash(q)) >= 8]
    ok = bool(qs) and all(q in source for q in qs)
    (stage1 if ok else dropped).append((kind, cs, d))
print(f"   1차 인용 정합성: 통과 {len(stage1)} / 탈락 {len(dropped)}")

# 1.5차: 정답이 실질적 내용을 담고 있는지 (표의 빈 셀을 근거로 한 문항 제거)
EMPTY_ANSWER = re.compile(r"^\s*(빈칸|공란|없음|해당\s*없음|N/?A|-|비어\s*있(음|다))\s*\.?\s*$", re.I)

def has_substance(answer: str) -> bool:
    """표의 빈 셀을 물어 '빈칸'이 정답이 되는 문항은 검색 성능을 측정하지 못하므로 제외."""
    return bool(answer and not EMPTY_ANSWER.match(answer.strip()) and len(answer.strip()) >= 2)

before = len(stage1)
stage1 = [(k, cs, d) for k, cs, d in stage1 if has_substance(d.answer)]
print(f"   1.5차 정답 실질성: 통과 {len(stage1)} / 탈락 {before - len(stage1)}")


class SelfContained(BaseModel):
    ok: bool = Field(description="질문만 단독으로 읽고 무엇을 묻는지 특정되면 True")
    reason: str

checker = ChatPromptTemplate.from_messages([
    ("system", "질문의 자기완결성을 평가합니다. 질문만 단독으로 전달했을 때 의도가 특정되는지 판단하세요. "
               "'이 문서', '해당 절차' 같은 모호한 표현이 있거나 주어가 빠졌으면 False입니다."),
    ("human", "질문: {question}"),
]) | ChatOpenAI(model=JUDGE, temperature=0).with_structured_output(SelfContained)

stage2 = [(k, cs, d) for k, cs, d in stage1 if checker.invoke({"question": d.question}).ok]
print(f"   2차 자기완결성: 최종 통과 {len(stage2)}")

# ---------- 5. 무답변 문항 자동 검증 ----------
# 1단계 파싱으로 회수된 텍스트가 3.8배 늘었으므로, 예전 무답변 후보가
# 이제는 문서에서 답이 나올 수 있다. 검색 + 심사로 '정말 답이 없는지' 확인한다.
print("\n[5/5] 무답변 후보 검증")

class Unanswerable(BaseModel):
    unanswerable: bool = Field(description="아래 문서 조각들만으로 질문에 답할 수 없으면 True")
    reason: str

verifier = ChatPromptTemplate.from_messages([
    ("system", "주어진 문서 조각만으로 질문에 답할 수 있는지 판정합니다. 부분적으로라도 답이 되면 False입니다."),
    ("human", "질문: {q}\n\n문서 조각:\n{ctx}"),
]) | ChatOpenAI(model=JUDGE, temperature=0).with_structured_output(Unanswerable)

NEGATIVE_CANDIDATES = [
    "Hmall 협력사 담당 MD의 직통 전화번호는 몇 번인가?",
    "현대백화점 오프라인 매장 입점 신청 절차와 제출 서류는 무엇인가?",
    "Hmall의 2025년 연간 총 거래액 목표는 얼마인가?",
    "쇼라 방송 출연 쇼호스트의 회당 출연료는 얼마인가?",
    "Hmall 협력사 시스템의 서버 장애 시 비상 연락망은 무엇인가?",
    "Hmall에는 총 몇 개의 오프라인 매장이 존재하는가?",
]

negatives = []
for q in NEGATIVE_CANDIDATES:
    ctx = "\n---\n".join(d.page_content for d in vectorstore.similarity_search(q, k=5))
    v = verifier.invoke({"q": q, "ctx": ctx})
    mark = "✅ 무답변 확정" if v.unanswerable else "❌ 답변 가능 → 제외"
    print(f"   {mark}: {q}")
    if v.unanswerable:
        negatives.append({
            "question": q,
            "ground_truth": "제공된 문서에서 정보를 찾을 수 없습니다.",
            "quote": "N/A",
            "question_type": "무답변",
            "source_context": "문서 내 해당 정보 없음",
        })
    if len(negatives) >= TARGET["무답변"]:
        break

# ---------- 6. 최종 구성 및 저장 ----------
final = []
for kind in ["단일홉", "멀티홉", "표/절차", "부정/예외", "시점/기한"]:
    picked = [x for x in stage2 if x[0] == kind][:TARGET[kind]]
    if len(picked) < TARGET[kind]:
        print(f"   ⚠️ {kind}: 목표 {TARGET[kind]}개 중 {len(picked)}개만 확보")
    for _, cs, d in picked:
        final.append({
            "question": d.question,
            "ground_truth": d.answer,
            "quotes": d.quotes,
            "question_type": kind,
            "difficulty": d.difficulty,
            "source": cs[0].metadata.get("source", "?"),
            "page": cs[0].metadata.get("page"),
            "source_context": "\n---\n".join(c.page_content for c in cs),
        })
final.extend(negatives)

with open("golden_dataset.json", "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=4)

from collections import Counter
print(f"\n🎉 골든 데이터셋 {len(final)}문항 저장 → golden_dataset.json")
print("   ", Counter(x["question_type"] for x in final))

---
# 3단계: RAG 검색 및 생성 워크플로우

Chroma 리트리버(k=4) → 프롬프트 → LLM → 문자열 파싱의 LCEL 체인.

프롬프트에 이 문서 집합 고유의 규칙 두 가지를 넣었다.

1. **무답변 방어** — 근거에 없으면 지어내지 말고 정해진 문장으로 거부
2. **표 행 조건 준수** — 질문이 지정한 행 조건을 모두 만족하는 행에서 값을 읽고,
   구분·하위구분이 다른 행의 값을 섞지 않도록 명시

검색된 청크는 `[출처: 파일명 p.페이지]` 를 붙여 합치고, 답변 끝에 출처를 남기게 했다.


In [ ]:
# =====================================================================
# 3단계: RAG 검색 및 생성 워크플로우
# =====================================================================
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. 벡터스토어 로드 및 리트리버 구성
vectorstore = Chroma(persist_directory=CHROMA_PATH,
                     embedding_function=embeddings,
                     collection_name=COLLECTION_NAME)

# k=4: 1단계 파싱이 표를 '마크다운 표 + 행 풀어쓰기'로 함께 남기므로
# 같은 표의 두 표현이 함께 회수될 여지를 두되, 컨텍스트가 과하게 길어지지 않는 선.
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# 2. 생성 모델
llm = ChatOpenAI(model=MODEL, temperature=0)

# 3. 프롬프트 — 문서 기반 답변 + 무답변 가이드 + 표 인용 규칙
template = """당신은 제공된 규정 및 매뉴얼 문서를 근거로 정확하게 답변하는 RAG 어시스턴트입니다.

아래 [참고 문서]만을 근거로 사용자 질문에 답하세요.

답변 규칙:
1. 참고 문서에서 답을 찾을 수 없으면 추측하지 말고 정확히 이렇게 답하세요:
   "제공된 문서에서 정보를 찾을 수 없습니다."
2. 표에서 값을 읽을 때는 질문이 지정한 **행 조건을 모두 만족하는 행**을 찾아 답하세요.
   구분·하위구분이 다른 행의 값을 섞어 쓰지 마세요.
3. 금액·기간·수수료율·URL·날짜는 문서에 적힌 표기 그대로 인용하세요.
4. 근거가 된 내용이 어느 문서·페이지인지 답변 끝에 (출처: 파일명 p.페이지) 형식으로 덧붙이세요.

[참고 문서]
{context}

[사용자 질문]
{question}

[답변]:"""
prompt = ChatPromptTemplate.from_template(template)


def format_docs(docs):
    """검색된 청크를 출처 정보와 함께 하나의 문자열로 합친다."""
    parts = []
    for d in docs:
        src = d.metadata.get("source", "알 수 없음")
        page = d.metadata.get("page", "?")
        parts.append(f"[출처: {src} p.{page}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)


# 4. LCEL 체인
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. 동작 확인 — 표 행/열 매핑과 무답변 방어를 각각 확인
for test_query in [
    "현대백화점탭 투뎁스 D_팝업 광고의 단가와 구좌수는?",
    "Hmall에는 총 몇 개의 오프라인 매장이 존재하는가?",
]:
    print("=" * 70)
    print(f"📌 질문: {test_query}")
    print(f"🤖 답변:\n{rag_chain.invoke(test_query)}\n")

---
# 4단계: RAG 성능 평가

### 지표 구성

| 지표 | 출처 | 측정 대상 |
|---|---|---|
| faithfulness | Ragas | 답변이 검색된 근거에 충실한가 (환각 여부) |
| answer_relevancy | Ragas | 답변이 질문에 실제로 답하는가 |
| context_precision | Ragas | 검색된 청크 중 실제로 쓸모 있는 비율 |
| context_recall | Ragas | 정답에 필요한 근거를 놓치지 않았는가 |
| **무답변 방어율** | 커스텀 | 문서에 없는 질문을 지어내지 않고 거부했는가 |
| **표 행/열 매핑 정확도** | 커스텀 | 조건에 맞는 '그 행'의 값을 골랐는가 |

커스텀 2지표가 이 프로젝트의 핵심이다. 특히 표 행/열 매핑은 1단계 파서 교체가
실제로 효과가 있었는지를 직접 측정한다.

### 구현 주의사항

설치된 **ragas 0.4.3은 API가 바뀌어** `from ragas.metrics import faithfulness` 가 동작하지 않는다.
메트릭 클래스를 각 모듈에서 직접 import 해야 한다.
또 심사위원 모델이 ragas 내부 기본값인 `temperature=0.01` 을 거부하므로
`LangchainLLMWrapper(..., bypass_temperature=True)` 로 감싼다.


In [ ]:
# =====================================================================
# 4단계: RAG 성능 평가 파이프라인
# ---------------------------------------------------------------------
# Ragas 표준 4지표 + 이 프로젝트 고유의 커스텀 2지표로 평가한다.
#  - 무답변 방어율   : 문서에 없는 질문에 지어내지 않고 거부했는가
#  - 표 행/열 매핑   : 표에서 조건에 맞는 '그 행'의 값을 골라냈는가 (1단계 개선의 직접 검증)
#
# 주의: ragas 0.4.x는 API가 바뀌어 `from ragas.metrics import faithfulness` 가 동작하지 않는다.
#       클래스를 직접 import 하고, temperature를 지원하지 않는 최신 모델을 심사위원으로
#       쓰기 위해 LangchainLLMWrapper(bypass_temperature=True) 로 감싼다.
# =====================================================================
import json
import pandas as pd
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics._faithfulness import Faithfulness
from ragas.metrics._answer_relevance import ResponseRelevancy
from ragas.metrics._context_precision import LLMContextPrecisionWithReference
from ragas.metrics._context_recall import LLMContextRecall

# ---------- 1. 골든 데이터셋 로드 ----------
df_gold = pd.read_json(GOLDEN_PATH)
print(f"📋 골든 데이터셋 {len(df_gold)}문항")
print(df_gold["question_type"].value_counts().to_string())

# ---------- 2. RAG 실행 결과 수집 ----------
print("\nRAG 답변 생성 및 컨텍스트 수집 중...")
samples, answers = [], []
for _, row in df_gold.iterrows():
    q = row["question"]
    answer = rag_chain.invoke(q)
    contexts = [d.page_content for d in retriever.invoke(q)]
    answers.append(answer)
    samples.append(SingleTurnSample(
        user_input=q,
        response=answer,
        retrieved_contexts=contexts,
        reference=row["ground_truth"],
    ))
print(f"✅ {len(samples)}문항 수집 완료")

# ---------- 3. Ragas 표준 지표 ----------
print("\nRagas 표준 지표 평가 중...")
eval_llm = LangchainLLMWrapper(ChatOpenAI(model=JUDGE), bypass_temperature=True)
eval_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model=EMBEDDING))

standard_result = evaluate(
    dataset=EvaluationDataset(samples=samples),
    metrics=[Faithfulness(), ResponseRelevancy(),
             LLMContextPrecisionWithReference(), LLMContextRecall()],
    llm=eval_llm,
    embeddings=eval_emb,
)
df_result = standard_result.to_pandas()
df_result["question_type"] = df_gold["question_type"].values

# ---------- 4. 커스텀 지표 ----------
print("커스텀 지표(무답변 방어 / 표 행·열 매핑) 평가 중...")
judge_llm = ChatOpenAI(model=JUDGE)

class RefusalEvaluation(BaseModel):
    is_refused_correctly: int = Field(
        description="문서에 없는 질문에 지어내지 않고 '정보를 찾을 수 없다'고 올바르게 거부했으면 1, 아니면 0")

class TabularMappingEvaluation(BaseModel):
    score: int = Field(
        description="질문이 지정한 행 조건에 해당하는 표의 값을 정확히 골라 답했으면 1, "
                    "다른 행의 값을 가져왔거나 값이 틀렸으면 0")

refusal_chain = ChatPromptTemplate.from_messages([
    ("system", "환각 방지 및 거부 답변 적절성 심사위원"),
    ("user", "질문: {question}\n정답: {ground_truth}\n시스템 답변: {answer}\n올바르게 거부했는가?"),
]) | judge_llm.with_structured_output(RefusalEvaluation)

tabular_chain = ChatPromptTemplate.from_messages([
    ("system", "표 형태 매뉴얼의 행/열 매핑 정확도 심사위원. "
               "질문이 지정한 행 조건(구분/하위구분/항목명 등)에 맞는 값을 답했는지만 본다."),
    ("user", "질문: {question}\n정답: {ground_truth}\n시스템 답변: {answer}\n"
             "표의 행/열 대응을 정확히 매칭했는가?"),
]) | judge_llm.with_structured_output(TabularMappingEvaluation)

refusal_scores, tabular_scores = [], []
for (_, row), ans in zip(df_gold.iterrows(), answers):
    kind = row["question_type"]
    payload = {"question": row["question"], "ground_truth": row["ground_truth"], "answer": ans}
    refusal_scores.append(refusal_chain.invoke(payload).is_refused_correctly if kind == "무답변" else None)
    tabular_scores.append(tabular_chain.invoke(payload).score if kind == "표/절차" else None)

df_result["custom_refusal_score"] = refusal_scores
df_result["custom_tabular_mapping_score"] = tabular_scores

# ---------- 5. 결과 출력 및 저장 ----------
cols = ["question", "question_type", "faithfulness", "answer_relevancy",
        "llm_context_precision_with_reference", "context_recall",
        "custom_refusal_score", "custom_tabular_mapping_score"]
cols = [c for c in cols if c in df_result.columns]

print("\n===== 문항별 평가 결과 =====")
display(df_result[cols])

print("\n===== 지표 요약 =====")
summary = {
    "faithfulness":        df_result["faithfulness"].mean(),
    "answer_relevancy":    df_result["answer_relevancy"].mean(),
    "context_precision":   df_result["llm_context_precision_with_reference"].mean(),
    "context_recall":      df_result["context_recall"].mean(),
    "무답변 방어율":        df_result["custom_refusal_score"].dropna().mean(),
    "표 행/열 매핑 정확도": df_result["custom_tabular_mapping_score"].dropna().mean(),
}
for k, v in summary.items():
    print(f"  {k:22s} {v:.4f}" if pd.notna(v) else f"  {k:22s} N/A")

print("\n===== 유형별 faithfulness =====")
print(df_result.groupby("question_type")["faithfulness"].mean().to_string())

output_csv = "rag_evaluation_final_result.csv"
df_result.to_csv(output_csv, index=False, encoding="utf-8-sig")
print(f"\n💾 평가 결과 저장 → {output_csv}")

---
# 5단계: Streamlit 챗봇

아래 셀의 내용은 **`app.py` 파일로 저장되어 있다.** 실행:

```bash
streamlit run app.py
```

3단계와 동일한 리트리버·프롬프트를 쓰되, `@st.cache_resource` 로 체인을 캐싱해
매 입력마다 벡터스토어를 다시 로드하지 않게 했다.
답변과 함께 참고한 청크를 출처·페이지와 같이 펼쳐볼 수 있다.


In [ ]:
# =====================================================================
# 5단계: Streamlit 챗봇
#   실행:  streamlit run app.py
# =====================================================================
import os
import streamlit as st
from dotenv import load_dotenv, find_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL     = os.environ["OPENAI_DEFAULT_MODEL"]
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]

st.set_page_config(page_title="Hmall 협력사 RAG 챗봇", page_icon="🤖", layout="centered")
st.title("📖 Hmall 협력사 규정·매뉴얼 어시스턴트")
st.markdown("협력사 운영 안내서, 광고 상품 소개서, 입점 절차 등 7개 문서를 근거로 답변합니다.")


@st.cache_resource
def init_rag():
    embeddings = OpenAIEmbeddings(model=EMBEDDING)
    vectorstore = Chroma(persist_directory="./chroma_db",
                         embedding_function=embeddings,
                         collection_name="langchain")
    retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

    template = """당신은 제공된 규정 및 매뉴얼 문서를 근거로 정확하게 답변하는 RAG 어시스턴트입니다.

아래 [참고 문서]만을 근거로 사용자 질문에 답하세요.

답변 규칙:
1. 참고 문서에서 답을 찾을 수 없으면 추측하지 말고 정확히 이렇게 답하세요:
   "제공된 문서에서 정보를 찾을 수 없습니다."
2. 표에서 값을 읽을 때는 질문이 지정한 **행 조건을 모두 만족하는 행**을 찾아 답하세요.
   구분·하위구분이 다른 행의 값을 섞어 쓰지 마세요.
3. 금액·기간·수수료율·URL·날짜는 문서에 적힌 표기 그대로 인용하세요.
4. 근거가 된 내용이 어느 문서·페이지인지 답변 끝에 (출처: 파일명 p.페이지) 형식으로 덧붙이세요.

[참고 문서]
{context}

[사용자 질문]
{question}

[답변]:"""

    def format_docs(docs):
        return "\n\n---\n\n".join(
            f"[출처: {d.metadata.get('source', '알 수 없음')} p.{d.metadata.get('page', '?')}]\n{d.page_content}"
            for d in docs
        )

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | ChatPromptTemplate.from_template(template)
        | ChatOpenAI(model=MODEL, temperature=0)
        | StrOutputParser()
    )
    return chain, retriever


rag_chain, retriever = init_rag()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

if user_input := st.chat_input("궁금한 점을 물어보세요 (예: 현대백화점탭 투뎁스 D_팝업 광고 단가는?)"):
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.markdown(user_input)

    with st.chat_message("assistant"):
        with st.spinner("문서를 검색하고 답변을 생성하는 중..."):
            retrieved_docs = retriever.invoke(user_input)
            response = rag_chain.invoke(user_input)

        with st.expander("🔍 참고한 문서 청크 보기"):
            for i, doc in enumerate(retrieved_docs):
                st.markdown(f"**[참고 {i+1}]** {doc.metadata.get('source', '?')} p.{doc.metadata.get('page', '?')}")
                st.text(doc.page_content[:400] + ("..." if len(doc.page_content) > 400 else ""))
                st.divider()

        st.markdown(response)

    st.session_state.messages.append({"role": "assistant", "content": response})